# Qwen3-VL Mind2Web Ablations: HTML-Only vs Image-Only
# Model: Qwen/Qwen3-VL-8B-Thinking | Dataset: osunlp/Multimodal-Mind2Web (test_website)
#
# Ablations:
#   1. Zero-Shot HTML-only  — cleaned HTML, no screenshot
#   2. Zero-Shot Image-only — screenshot only, no HTML
#   3. Few-Shot  HTML-only  — 3 in-context examples (HTML only)
#   4. Few-Shot  Image-only — 3 in-context examples (images only)
#
# Scoring: per-candidate log-likelihood (length-normalised)
# VRAM: ~18-22 GB (bf16 8B model)


In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Enter your HF token:")
#

## Cell 1: INSTALL PACKAGES


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate datasets pillow torch qwen-vl-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 65.3 MB/s eta 0:00:00


## Download Mind2Web Data to $LOCAL
Download the full Mind2Web dataset (JSONL and images) directly to the assigned node's $LOCAL storage. This avoids Hugging Face cache and ensures all data is local.

In [ ]:
import os
import re
import gdown

# Use $LOCAL or fallback to /tmp
LOCAL = os.environ.get("LOCAL") or os.environ.get("local") or "/tmp/abhyanka_local_cache"
os.makedirs(LOCAL, exist_ok=True)

# Mind2Web data (JSONL + images) direct download link (replace with actual link if needed)
mind2web_drive_url = "https://drive.google.com/file/d/1leHOcLv631qfOniHsI409CW1aC48cEt8/view?usp=sharing"

def normalize_drive_download_url(url: str) -> str:
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if not m:
        m = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
    if m:
        return f"https://drive.google.com/uc?id={m.group(1)}"
    return url

zip_path = os.path.join(LOCAL, "mind2web_data.zip")
extract_path = os.path.join(LOCAL, "project_data")
os.makedirs(extract_path, exist_ok=True)

download_url = normalize_drive_download_url(mind2web_drive_url)
print("Downloading Mind2Web data from:", download_url)
print("Zip path:", zip_path)
print("Extract path:", extract_path)

gdown.download(download_url, zip_path, quiet=False)

!unzip -qo $zip_path -d $extract_path
!rm -f $zip_path

print("Mind2Web data downloaded and extracted to:", extract_path)

## Cell 2: IMPORTS


In [1]:
import os
import json
import re
import time
import math
import gc
import traceback
from io import BytesIO
from collections import defaultdict, Counter
from datetime import datetime

import torch
import numpy as np
from PIL import Image
from datasets import load_dataset

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

/ocean/projects/cis260137p/abhyanka/.project/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'1'

## Cell 3: CONFIGURATION


In [ ]:
CONFIG = {
    "base_model_id": "Qwen/Qwen3-VL-8B-Thinking",
    "model_path": "/ocean/projects/cis260137p/abhyanka/outputs/qwen-vl-next-action-sft/checkpoint-6545",
    "processor_path": "/ocean/projects/cis260137p/abhyanka/outputs/qwen-vl-next-action-sft",
    "dataset_id": "osunlp/Multimodal-Mind2Web",
    "eval_split": "test_website",
    "train_split": "train",
    "max_html_chars": 12000,
    "max_new_tokens": 512,
    "sanity_check_n": 10,
    "local_dir": os.environ.get("LOCAL", "/tmp/abhyanka_local_cache"),
    "output_dir": "/ocean/projects/cis260137p/abhyanka/outputs/qwen3_vl_mind2web_ablations",
    "use_bf16": True,
    "use_flash_attn": True,
    "use_candidate_scoring": True,
    "scoring_max_new_tokens": 10,
    "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
}

CONFIG["dataset_cache_dir"] = os.path.join(CONFIG["local_dir"], "hf_datasets")
CONFIG["hub_cache_dir"] = os.path.join(CONFIG["local_dir"], "hf_hub")
CONFIG["hf_home"] = os.path.join(CONFIG["local_dir"], "hf_home")

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["dataset_cache_dir"], exist_ok=True)
os.makedirs(CONFIG["hub_cache_dir"], exist_ok=True)
os.makedirs(CONFIG["hf_home"], exist_ok=True)

# Force all HF caches into LOCAL-backed storage to avoid home quota issues.
os.environ["HF_HOME"] = CONFIG["hf_home"]
os.environ["HF_DATASETS_CACHE"] = CONFIG["dataset_cache_dir"]
os.environ["HF_HUB_CACHE"] = CONFIG["hub_cache_dir"]
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Dataset cache: {CONFIG['dataset_cache_dir']}")
print(f"Hub cache:     {CONFIG['hub_cache_dir']}")

with open(os.path.join(CONFIG["output_dir"], "run_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)


Dataset cache: /local/hf_datasets
Hub cache:     /local/hf_hub


## Cell 4: LOAD DATASET


In [5]:
print("Downloading/loading Mind2Web into LOCAL cache...")
print(f"Cache dir: {CONFIG['dataset_cache_dir']}")

ds_test = load_dataset(
    CONFIG["dataset_id"],
    split=CONFIG["eval_split"],
    cache_dir=CONFIG["dataset_cache_dir"],
)
ds_train = load_dataset(
    CONFIG["dataset_id"],
    split=CONFIG["train_split"],
    cache_dir=CONFIG["dataset_cache_dir"],
)

print(f"Loaded test examples: {len(ds_test)}")
print(f"Loaded train examples: {len(ds_train)}")


Downloading/loading Mind2Web into LOCAL cache...
Cache dir: /local/hf_datasets


/ocean/projects/cis260137p/abhyanka/.project/lib64/python3.11/site-packages/huggingface_hub/file_download.py:731: UserWarning: Not enough free disk space to download the file. The expected file size is: 316.47 MB. The target location /jet/home/abhyanka/.cache/huggingface/hub/datasets--osunlp--Multimodal-Mind2Web/blobs only has 0.00 MB free disk space.
  warnings.warn(


RuntimeError: Data processing error: File reconstruction error: IO Error: Disk quota exceeded (os error 122)

## Cell 5: INSPECT DATASET SCHEMA


In [ ]:
print("Dataset Features")
print(ds_test.features)
print()

ex=ds_test[0]

print("Example Keys")
for k,v in ex.items():
    if isinstance(v,str):
        print(f"{k}: str,len={len(v)},preview={v[:120]}...")
    elif isinstance(v,list):
        first=str(v[0])[:120] if len(v)>0 else "EMPTY"
        print(f"{k}: list,len={len(v)},first={first}...")
    elif isinstance(v,Image.Image):
        print(f"{k}: PIL.Image,size={v.size},mode={v.mode}")
    elif isinstance(v,dict):
        print(f"{k}: dict,keys={list(v.keys())}")
    else:
        print(f"{k}: {type(v).__name__},value={str(v)[:120]}")

print()
print("operation field")
op=json.loads(ex["operation"]) if isinstance(ex["operation"],str) else ex["operation"]
print(json.dumps(op,indent=2))

print()
print("action_reprs first 5")
for i,at in enumerate(ex["action_reprs"][:5]):
    print(f"[{i}] {at}")

print()
print("target_action_index")
print(ex["target_action_index"])

print()
print("target_action_reprs")
print(ex["target_action_reprs"])

Dataset Features
{'action_uid': Value('string'), 'raw_html': Value('string'), 'cleaned_html': Value('string'), 'operation': Value('string'), 'pos_candidates': List(Value('string')), 'neg_candidates': List(Value('string')), 'website': Value('string'), 'domain': Value('string'), 'subdomain': Value('string'), 'annotation_id': Value('string'), 'confirmed_task': Value('string'), 'screenshot': Image(mode=None, decode=True), 'action_reprs': List(Value('string')), 'target_action_index': Value('string'), 'target_action_reprs': Value('string')}

Example Keys
action_uid: str,len=36,preview=79c4a963-4aa9-49c1-9257-6b0d5069c551...
raw_html: str,len=85102,preview=<html backend_node_id="113">
  <body backend_node_id="188">
    <div backend_node_id="189">
      <div backend_node_id="...
cleaned_html: str,len=85102,preview=<html backend_node_id="113">
  <body backend_node_id="188">
    <div backend_node_id="189">
      <div backend_node_id="...
operation: str,len=52,preview={"original_op": "CLICK", "va

## Cell 6: PARSE DATASET HELPERS


In [ ]:
def parse_operation(op_field):
    if isinstance(op_field, str):
        return json.loads(op_field)
    return op_field

def get_op_type(op_dict):
    return op_dict.get("op", "CLICK").upper()

def get_op_value(op_dict):
    return op_dict.get("value", "")

def truncate_html(html_str, max_chars):
    if len(html_str) <= max_chars:
        return html_str, False
    half = max_chars // 2
    return html_str[:half] + "\n... [TRUNCATED] ...\n" + html_str[-half:], True

def get_candidate_actions(example):
    return example["action_reprs"]

def get_target_index(example):
    idx_str = example["target_action_index"]
    try:
        return int(idx_str)
    except (ValueError, TypeError):
        return -1

def get_target_repr(example):
    val = example["target_action_reprs"]
    # Guard: if the field is a list, take the first element
    if isinstance(val, list):
        return val[0] if len(val) > 0 else ""
    return val

def get_screenshot(example):
    img = example["screenshot"]
    if isinstance(img, Image.Image):
        return img
    if isinstance(img, dict) and "bytes" in img:
        return Image.open(BytesIO(img["bytes"]))
    if isinstance(img, dict) and "path" in img:
        return Image.open(img["path"])
    return None

def get_task_id(example):
    return example["annotation_id"]

def downscale_image(img, max_size):
    """Downscale an image so its longest side is at most max_size."""
    if img is None:
        return None
    w, h = img.size
    if max(w, h) <= max_size:
        return img
    scale = max_size / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


ex = ds_test[0]
op = parse_operation(ex["operation"])
img = get_screenshot(ex)

## Cell 7: SELECT FEW-SHOT EXAMPLES


In [ ]:
def find_fewshot_examples(train_ds, target_ops=["CLICK", "TYPE", "SELECT"], exclude_websites=None):
    if exclude_websites is None:
        exclude_websites = set()
    found = {}
    for i in range(len(train_ds)):
        if len(found) == len(target_ops):
            break
        ex = train_ds[i]
        op = parse_operation(ex["operation"])
        op_type = get_op_type(op)
        if op_type in target_ops and op_type not in found:
            ws = ex.get("website", "")
            if ws not in exclude_websites:
                candidates = get_candidate_actions(ex)
                target_idx = get_target_index(ex)
                if target_idx >= 0 and target_idx < len(candidates) and len(candidates) > 1:
                    img = get_screenshot(ex)
                    if img is not None:
                        found[op_type] = {
                            "index": i,
                            "example": ex,
                            "op_type": op_type,
                            "image": img,
                        }
    return found

test_websites = set(ds_test.unique("website")) if "website" in ds_test.column_names else set()
print(f"Test websites to exclude from few-shot: {len(test_websites)}")

fewshot_examples = find_fewshot_examples(ds_train, exclude_websites=test_websites)
print(f"Found few-shot examples for ops: {list(fewshot_examples.keys())}")
for op_type, info in fewshot_examples.items():
    ex = info["example"]
    print(f"  {op_type}: website={ex['website']}, task={ex['confirmed_task'][:80]}, "
          f"target_idx={get_target_index(ex)}, n_cands={len(get_candidate_actions(ex))}")

Test websites to exclude from few-shot: 10
Test websites to exclude from few-shot: 10
Found few-shot examples for ops: ['CLICK', 'TYPE', 'SELECT']
  CLICK: website=united, task=rent a car in Brooklyn - Central, NY on from April 9 to April 15., target_idx=0, n_cands=7
  TYPE: website=united, task=rent a car in Brooklyn - Central, NY on from April 9 to April 15., target_idx=1, n_cands=7
  SELECT: website=ign, task=Show computer game reviews sorted by score., target_idx=2, n_cands=4
Found few-shot examples for ops: ['CLICK', 'TYPE', 'SELECT']
  CLICK: website=united, task=rent a car in Brooklyn - Central, NY on from April 9 to April 15., target_idx=0, n_cands=7
  TYPE: website=united, task=rent a car in Brooklyn - Central, NY on from April 9 to April 15., target_idx=1, n_cands=7
  SELECT: website=ign, task=Show computer game reviews sorted by score., target_idx=2, n_cands=4


## Cell 8: PROMPT CONSTRUCTION


In [ ]:
_SUFFIX = (
    "Your job is to select the single best action that accomplishes the user's task. "
    "Do NOT invent new actions. You MUST choose from the provided candidates only.\n\n"
    "After your reasoning, you MUST end your response with exactly:\nAnswer: <index>\n"
    "where <index> is the zero-based integer index of your chosen action from the candidate list."
)
SYSTEM_PROMPTS = {
    "both":       "You are a web navigation agent. You are given a screenshot of a webpage, the cleaned HTML of the page, and a list of candidate actions. " + _SUFFIX,
    "html_only":  "You are a web navigation agent. You are given the cleaned HTML of a webpage and a list of candidate actions. " + _SUFFIX,
    "image_only": "You are a web navigation agent. You are given a screenshot of a webpage and a list of candidate actions. " + _SUFFIX,
}
# Keep backward-compat alias
SYSTEM_PROMPT = SYSTEM_PROMPTS["both"]

def build_candidate_str(candidates):
    lines = []
    for i, c in enumerate(candidates):
        lines.append(f"[{i}] {c}")
    return "\n".join(lines)

def build_zeroshot_prompt(example, max_html_chars):
    html_raw = example["cleaned_html"]
    html_text, was_truncated = truncate_html(html_raw, max_html_chars)
    candidates = get_candidate_actions(example)
    cand_str = build_candidate_str(candidates)
    task = example["confirmed_task"]

    trunc_note = ""
    if was_truncated:
        trunc_note = f" (truncated from {len(html_raw)} to {max_html_chars} chars)"

    text = (
        f"Task: {task}\n\n"
        f"Cleaned HTML{trunc_note}:\n{html_text}\n\n"
        f"Candidate Actions:\n{cand_str}\n\n"
        f"Select the correct action index. Respond with:\nAnswer: <index>"
    )
    return text, html_text

def build_fewshot_prefix(fewshot_examples_dict, max_html_chars):
    parts = []
    for op_type in ["CLICK", "TYPE", "SELECT"]:
        if op_type not in fewshot_examples_dict:
            continue
        info = fewshot_examples_dict[op_type]
        ex = info["example"]
        html_text, _ = truncate_html(ex["cleaned_html"], max_html_chars)
        candidates = get_candidate_actions(ex)
        cand_str = build_candidate_str(candidates)
        target_idx = get_target_index(ex)
        task = ex["confirmed_task"]

        parts.append(
            f"--- Example ({op_type}) ---\n"
            f"Task: {task}\n\n"
            f"Cleaned HTML:\n{html_text}\n\n"
            f"Candidate Actions:\n{cand_str}\n\n"
            f"Answer: {target_idx}"
        )
    return "\n\n".join(parts)

def build_fewshot_prefix_nohtml(fewshot_examples_dict):
    """Few-shot prefix without HTML — used for image_only mode."""
    parts = []
    for op_type in ["CLICK", "TYPE", "SELECT"]:
        if op_type not in fewshot_examples_dict:
            continue
        info = fewshot_examples_dict[op_type]
        ex = info["example"]
        candidates = get_candidate_actions(ex)
        cand_str = build_candidate_str(candidates)
        target_idx = get_target_index(ex)
        task = ex["confirmed_task"]

        parts.append(
            f"--- Example ({op_type}) ---\n"
            f"Task: {task}\n\n"
            f"Candidate Actions:\n{cand_str}\n\n"
            f"Answer: {target_idx}"
        )
    return "\n\n".join(parts)

def build_fewshot_prompt(example, fewshot_prefix, max_html_chars):
    html_raw = example["cleaned_html"]
    html_text, was_truncated = truncate_html(html_raw, max_html_chars)
    candidates = get_candidate_actions(example)
    cand_str = build_candidate_str(candidates)
    task = example["confirmed_task"]

    trunc_note = ""
    if was_truncated:
        trunc_note = f" (truncated from {len(html_raw)} to {max_html_chars} chars)"

    text = (
        f"{fewshot_prefix}\n\n"
        f"--- Now predict ---\n"
        f"Task: {task}\n\n"
        f"Cleaned HTML{trunc_note}:\n{html_text}\n\n"
        f"Candidate Actions:\n{cand_str}\n\n"
        f"Select the correct action index. Respond with:\nAnswer: <index>"
    )
    return text, html_text

fewshot_prefix = build_fewshot_prefix(fewshot_examples, CONFIG["max_html_chars"])
fewshot_prefix_nohtml = build_fewshot_prefix_nohtml(fewshot_examples)
print(f"Few-shot prefix length (with HTML): {len(fewshot_prefix)} chars")
print(f"Few-shot prefix length (no HTML):   {len(fewshot_prefix_nohtml)} chars")

print("\n=== Zero-shot prompt preview (first example, first 500 chars) ===")
zs_prompt, _ = build_zeroshot_prompt(ds_test[0], CONFIG["max_html_chars"])
print(zs_prompt[:500])


Few-shot prefix length (with HTML): 37301 chars
Few-shot prefix length (no HTML):   1190 chars

=== Zero-shot prompt preview (first example, first 500 chars) ===
Task: What are the romantic reggae musics from BCD Studio that can be used in tik tok series in andorra

Cleaned HTML (truncated from 85102 to 12000 chars):
<html backend_node_id="113">
  <body backend_node_id="188">
    <div backend_node_id="189">
      <div backend_node_id="190">
        <div backend_node_id="191">
          <header backend_node_id="192">
            <div backend_node_id="193">
              <a backend_node_id="194">
                <img backend_node_id="106" alt="Tiktok for b
Few-shot prefix length (with HTML): 37301 chars
Few-shot prefix length (no HTML):   1190 chars

=== Zero-shot prompt preview (first example, first 500 chars) ===
Task: What are the romantic reggae musics from BCD Studio that can be used in tik tok series in andorra

Cleaned HTML (truncated from 85102 to 12000 chars):
<html backend_node

## Cell 9: LOAD MODEL


In [ ]:
print("Loading model...")

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

attn_impl = "flash_attention_2" if CONFIG["use_flash_attn"] else "sdpa"
model_path = CONFIG["model_path"]
base_model_id = CONFIG["base_model_id"]

try:
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
        attn_implementation=attn_impl,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    if os.path.exists(os.path.join(model_path, "adapter_config.json")):
        model = PeftModel.from_pretrained(base_model, model_path)
        model = model.merge_and_unload()
        print(f"Loaded LoRA adapter from {model_path}")
    else:
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
            attn_implementation=attn_impl,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
    print(f"Model loaded with attn_implementation={attn_impl}")
except Exception as e:
    print(f"Failed with {attn_impl}: {e}")
    print("Falling back to sdpa...")
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
        attn_implementation="sdpa",
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    if os.path.exists(os.path.join(model_path, "adapter_config.json")):
        model = PeftModel.from_pretrained(base_model, model_path)
        model = model.merge_and_unload()
        print(f"Loaded LoRA adapter from {model_path}")
    else:
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            model_path,
            torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
            attn_implementation="sdpa",
            device_map="auto",
            low_cpu_mem_usage=True,
        )
    print("Model loaded with attn_implementation=sdpa")

processor = AutoProcessor.from_pretrained(CONFIG["processor_path"])
model.eval()
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")


Loading model...
Loading model...


config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Model device: cuda:0
Model dtype: torch.bfloat16
Model device: cuda:0
Model dtype: torch.bfloat16


## Cell 10: INFERENCE FUNCTIONS


In [ ]:
def build_messages_zeroshot(example, max_html_chars, input_mode="both"):
    """input_mode: 'html_only', 'image_only', or 'both'"""
    img = get_screenshot(example)
    content = []

    if input_mode == "image_only":
        # No HTML processing needed for image-only mode
        if img is not None:
            content.append({"type": "image", "image": img})
        minimal_prompt = (
            f"Task: {example['confirmed_task']}\n\n"
            f"Candidate Actions:\n{build_candidate_str(get_candidate_actions(example))}\n\n"
            f"Select the correct action index. Respond with:\nAnswer: <index>"
        )
        content.append({"type": "text", "text": minimal_prompt})
        used_html = ""
    else:
        text_prompt, used_html = build_zeroshot_prompt(example, max_html_chars)
        if input_mode == "both" and img is not None:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": text_prompt})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPTS[input_mode]}]},
        {"role": "user", "content": content},
    ]
    return messages, used_html

def build_messages_fewshot(example, fewshot_prefix_str, fewshot_prefix_nohtml_str,
                           fewshot_images, max_html_chars, input_mode="both"):
    """
    input_mode: 'html_only', 'image_only', or 'both'

    fewshot_prefix_str:        few-shot prefix WITH HTML  (for html_only / both)
    fewshot_prefix_nohtml_str: few-shot prefix WITHOUT HTML (for image_only)
    """
    img = get_screenshot(example)
    content = []

    if input_mode == "image_only":
        # Add few-shot images + test image
        for fs_img in fewshot_images:
            if fs_img is not None:
                content.append({"type": "image", "image": fs_img})
        if img is not None:
            content.append({"type": "image", "image": img})

        minimal_prompt = (
            f"{fewshot_prefix_nohtml_str}\n\n"
            f"--- Now predict ---\n"
            f"Task: {example['confirmed_task']}\n\n"
            f"Candidate Actions:\n{build_candidate_str(get_candidate_actions(example))}\n\n"
            f"Select the correct action index. Respond with:\nAnswer: <index>"
        )
        content.append({"type": "text", "text": minimal_prompt})
        used_html = ""
    else:
        text_prompt, used_html = build_fewshot_prompt(example, fewshot_prefix_str, max_html_chars)
        if input_mode == "both":
            for fs_img in fewshot_images:
                if fs_img is not None:
                    content.append({"type": "image", "image": fs_img})
            if img is not None:
                content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": text_prompt})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPTS[input_mode]}]},
        {"role": "user", "content": content},
    ]
    return messages, used_html

def _apply_chat_template(messages):
    try:
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        )
    except TypeError:
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
    inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}
    return inputs

def generate_response(messages, max_new_tokens):
    inputs = _apply_chat_template(messages)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    input_len = inputs["input_ids"].shape[1]
    output_ids = generated_ids[0][input_len:]
    return processor.tokenizer.decode(output_ids, skip_special_tokens=True)

def prepare_inputs_once(messages):
    return _apply_chat_template(messages)

def score_candidates_batched(cached_inputs, num_candidates):
    """Score all candidates using a single forward pass for the prompt prefix (KV cache),
    then a cheap per-candidate forward pass over just the answer tokens."""

    # Step 1: Run the prompt through the model ONCE and cache the KV states
    input_ids = cached_inputs["input_ids"]
    forward_inputs = {"input_ids": input_ids, "use_cache": True}
    if "attention_mask" in cached_inputs:
        forward_inputs["attention_mask"] = cached_inputs["attention_mask"]
    for key in ["pixel_values", "image_grid_thw", "pixel_values_videos", "video_grid_thw"]:
        if key in cached_inputs:
            forward_inputs[key] = cached_inputs[key]

    with torch.no_grad():
        base_outputs = model(**forward_inputs)
    past_key_values = base_outputs.past_key_values
    base_logits = base_outputs.logits  # [1, seq_len, vocab_size]

    prompt_len = input_ids.shape[1]

    scores = []
    for c in range(num_candidates):
        answer_text = f"Answer: {c}"
        answer_ids = processor.tokenizer.encode(answer_text, add_special_tokens=False)

        # First token of the answer is predicted from the last prompt position
        log_p_first = torch.log_softmax(base_logits[0, -1, :], dim=-1)
        log_probs = [log_p_first[answer_ids[0]].item()]

        if len(answer_ids) > 1:
            # Score remaining answer tokens using the cached KV
            remaining_ids = torch.tensor([answer_ids[:-1]], device=model.device)
            # Build attention mask covering prompt + answer tokens so far
            ext_mask = torch.ones(1, prompt_len + remaining_ids.shape[1],
                                  device=model.device,
                                  dtype=cached_inputs.get("attention_mask",
                                        torch.ones(1)).dtype)

            with torch.no_grad():
                cont_outputs = model(
                    input_ids=remaining_ids,
                    attention_mask=ext_mask,
                    past_key_values=past_key_values,
                    use_cache=False,
                )
            cont_logits = cont_outputs.logits  # [1, len(remaining_ids), vocab]
            for t_idx in range(cont_logits.shape[1]):
                log_p = torch.log_softmax(cont_logits[0, t_idx, :], dim=-1)
                log_probs.append(log_p[answer_ids[t_idx + 1]].item())

        scores.append(sum(log_probs) / len(log_probs))

    # Free KV cache memory
    del past_key_values, base_outputs, base_logits
    return scores

def run_inference_single(messages, num_candidates, use_scoring=True):
    if use_scoring and num_candidates <= 50:
        cached_inputs = prepare_inputs_once(messages)
        scores = score_candidates_batched(cached_inputs, num_candidates)
        del cached_inputs
        ranked = sorted(range(num_candidates), key=lambda i: scores[i], reverse=True)
        raw_output = json.dumps({"scores": {str(i): round(scores[i], 4) for i in range(num_candidates)}})
        return ranked[0], ranked[:3], raw_output, scores
    else:
        raw_output = generate_response(messages, CONFIG["max_new_tokens"])
        parsed = parse_answer(raw_output, num_candidates)
        return parsed, [parsed], raw_output, []

def parse_answer(text, num_candidates):
    for pat in [r"Answer:\s*(\d+)", r"answer:\s*(\d+)", r"\b(\d+)\s*$"]:
        m = re.search(pat, text)
        if m:
            idx = int(m.group(1))
            if 0 <= idx < num_candidates:
                return idx
    for n_str in reversed(re.findall(r"\b(\d+)\b", text)):
        idx = int(n_str)
        if 0 <= idx < num_candidates:
            return idx
    return -1

print("Inference functions defined.")

Inference functions defined.
Inference functions defined.


## Cell 11: METRIC COMPUTATION


In [ ]:
def compute_element_accuracy(pred_idx, gold_idx):
    return int(pred_idx == gold_idx)

def char_f1(pred_str, gold_str):
    """Character-level F1 using Counter for O(n) multiset intersection."""
    pred_counts = Counter(pred_str)
    gold_counts = Counter(gold_str)
    if len(pred_str) == 0 and len(gold_str) == 0:
        return 1.0
    if len(pred_str) == 0 or len(gold_str) == 0:
        return 0.0
    common = sum((pred_counts & gold_counts).values())
    precision = common / len(pred_str)
    recall = common / len(gold_str)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def extract_op_and_value_from_repr(action_repr):
    m = re.search(r"->\s*(CLICK|TYPE|SELECT)\s*(.*)", action_repr)
    if m:
        op_type = m.group(1).strip()
        value = m.group(2).strip()
        return op_type, value
    return action_repr.strip(), ""

def compute_operation_f1(pred_action_repr, gold_action_repr):
    pred_op, pred_val = extract_op_and_value_from_repr(pred_action_repr)
    gold_op, gold_val = extract_op_and_value_from_repr(gold_action_repr)
    pred_combined = f"{pred_op} {pred_val}".strip()
    gold_combined = f"{gold_op} {gold_val}".strip()
    return char_f1(pred_combined, gold_combined)

def compute_step_sr(pred_idx, gold_idx, pred_action_repr, gold_action_repr):
    if pred_idx != gold_idx:
        return 0
    f1 = compute_operation_f1(pred_action_repr, gold_action_repr)
    return int(f1 >= 0.9)

def compute_top3_accuracy(top3_indices, gold_idx):
    return int(gold_idx in top3_indices)

def compute_metrics(predictions):
    n = len(predictions)
    if n == 0:
        return {}
    ele_acc_sum = 0
    op_f1_sum = 0
    step_sr_sum = 0
    top3_sum = 0
    task_steps = defaultdict(list)
    for p in predictions:
        gold_idx = p["gold_target_index"]
        pred_idx = p["predicted_index"]
        candidates = p["candidate_actions"]
        gold_repr = p["gold_target_action"]
        pred_repr = candidates[pred_idx] if 0 <= pred_idx < len(candidates) else ""
        ea = compute_element_accuracy(pred_idx, gold_idx)
        of1 = compute_operation_f1(pred_repr, gold_repr)
        ssr = compute_step_sr(pred_idx, gold_idx, pred_repr, gold_repr)
        t3 = compute_top3_accuracy(p.get("top3_predicted_indices", [pred_idx]), gold_idx)
        ele_acc_sum += ea
        op_f1_sum += of1
        step_sr_sum += ssr
        top3_sum += t3
        p["metric_ele_acc"] = ea
        p["metric_op_f1"] = of1
        p["metric_step_sr"] = ssr
        p["metric_top3_acc"] = t3
        task_id = p.get("task_id", "unknown")
        task_steps[task_id].append(ssr)
    task_sr_sum = 0
    for task_id, steps in task_steps.items():
        task_sr_sum += int(all(s == 1 for s in steps))
    num_tasks = len(task_steps)
    metrics = {
        "element_accuracy": ele_acc_sum / n,
        "operation_f1": op_f1_sum / n,
        "step_success_rate": step_sr_sum / n,
        "task_success_rate": task_sr_sum / num_tasks if num_tasks > 0 else 0,
        "top3_accuracy": top3_sum / n,
        "num_examples": n,
        "num_tasks": num_tasks,
    }
    return metrics

print("Metric functions defined.")

Metric functions defined.
Metric functions defined.


## Cell 12: MAIN INFERENCE LOOP


In [ ]:
def run_ablation(dataset, ablation_name, mode="zero_shot", n_examples=None,
                 fewshot_prefix_str=None, fewshot_prefix_nohtml_str=None,
                 fewshot_images=None, input_mode="html_only",
                 checkpoint_every=100):
    """
    input_mode: "html_only" or "image_only"
    fewshot_prefix_str:        few-shot prefix WITH HTML
    fewshot_prefix_nohtml_str: few-shot prefix WITHOUT HTML
    checkpoint_every: save a recovery checkpoint every N examples (useful on Colab)
    """
    predictions = []
    total = len(dataset) if n_examples is None else min(n_examples, len(dataset))
    use_scoring = CONFIG["use_candidate_scoring"]

    print(f"\n{'='*60}")
    print(f"Running ablation: {ablation_name} ({mode}, input={input_mode})")
    print(f"Examples: {total}, Scoring: {use_scoring}")
    print(f"{'='*60}\n")

    start_time = time.time()

    for i in range(total):
        ex = dataset[i]
        candidates = get_candidate_actions(ex)
        gold_idx = get_target_index(ex)
        gold_repr = get_target_repr(ex)
        task_id = get_task_id(ex)
        html_raw = ex["cleaned_html"]

        if mode == "zero_shot":
            messages, used_html = build_messages_zeroshot(ex, CONFIG["max_html_chars"], input_mode=input_mode)
        else:
            messages, used_html = build_messages_fewshot(
                ex, fewshot_prefix_str, fewshot_prefix_nohtml_str,
                fewshot_images, CONFIG["max_html_chars"], input_mode=input_mode
            )

        try:
            pred_idx, top3, raw_output, scores = run_inference_single(
                messages, len(candidates), use_scoring=use_scoring
            )
        except Exception as e:
            print(f"  ERROR on example {i}: {e}")
            traceback.print_exc()
            pred_idx = -1
            top3 = [-1]
            raw_output = f"ERROR: {str(e)}"
            scores = []

        top3_actions = [candidates[j] if 0 <= j < len(candidates) else "INVALID" for j in top3]
        pred_action = candidates[pred_idx] if 0 <= pred_idx < len(candidates) else "INVALID"

        predictions.append({
            "example_index": i,
            "action_uid": ex["action_uid"],
            "task_id": task_id,
            "website": ex.get("website", ""),
            "confirmed_task": ex["confirmed_task"],
            "truncated_html_len": len(used_html),
            "original_html_len": len(html_raw),
            "was_truncated": len(used_html) < len(html_raw) if used_html else False,
            "num_candidates": len(candidates),
            "candidate_actions": candidates,
            "gold_target_index": gold_idx,
            "gold_target_action": gold_repr,
            "predicted_index": pred_idx,
            "predicted_action": pred_action,
            "top3_predicted_indices": top3,
            "top3_predicted_actions": top3_actions,
            "raw_model_output": raw_output[:2000],
        })

        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        eta = avg_time * (total - i - 1)
        correct_str = "✓" if pred_idx == gold_idx else "✗"
        print(f"  [{i+1}/{total}] {correct_str} pred={pred_idx} gold={gold_idx} "
              f"| {avg_time:.1f}s/ex, ETA: {eta/60:.1f}min")

        if i < 3:
            print(f"    task: {ex['confirmed_task'][:80]}")
            print(f"    pred: {pred_action[:80]}")
            print(f"    gold: {gold_repr[:80]}")

        if checkpoint_every and (i + 1) % checkpoint_every == 0:
            ckpt_path = os.path.join(CONFIG["output_dir"], f"{ablation_name}_ckpt_{i+1}.json")
            with open(ckpt_path, "w") as f:
                json.dump(predictions, f, indent=2, default=str)
            interim = compute_metrics(predictions)
            print(f"  ** Ckpt {i+1}: ele_acc={interim['element_accuracy']:.4f} "
                  f"op_f1={interim['operation_f1']:.4f} "
                  f"step_sr={interim['step_success_rate']:.4f} "
                  f"top3={interim['top3_accuracy']:.4f}")

    total_time = time.time() - start_time
    print(f"\nCompleted {total} examples in {total_time/60:.1f} min ({total_time/3600:.1f} hrs)")

    metrics = compute_metrics(predictions)
    metrics["ablation"] = ablation_name
    metrics["mode"] = mode
    metrics["input_mode"] = input_mode
    metrics["total_time_seconds"] = total_time

    return predictions, metrics

## Cell 13: SANITY CHECK (10 EXAMPLES, ZERO-SHOT)


In [ ]:
# Sanity check: 10 examples across html/image/both modes
for mode_name, i_mode in [("html_only", "html_only"), ("image_only", "image_only"), ("html_image", "both")]:
    preds, metrics = run_ablation(
        ds_test, f"sanity_{mode_name}", mode="zero_shot",
        n_examples=CONFIG["sanity_check_n"], input_mode=i_mode
    )
    print(f"\n=== Sanity [{mode_name}] ===")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print("\n=== Sample outputs ===")
    for p in preds[:2]:
        print(f"  Example {p['example_index']}:")
        print(f"    Task: {p['confirmed_task'][:80]}")
        print(f"    Gold={p['gold_target_index']}  Pred={p['predicted_index']}  Top3={p['top3_predicted_indices']}")


Keyword argument `enable_thinking` is not a valid argument for this processor and will be ignored.
Keyword argument `enable_thinking` is not a valid argument for this processor and will be ignored.



Running ablation: sanity_html_only (zero_shot, input=html_only)
Examples: 10, Scoring: True


Running ablation: sanity_html_only (zero_shot, input=html_only)
Examples: 10, Scoring: True

  [1/10] ✓ pred=0 gold=0 | 1.9s/ex, ETA: 0.3min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [label]   -> CLICK
  [1/10] ✓ pred=0 gold=0 | 1.9s/ex, ETA: 0.3min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [label]   -> CLICK
  [2/10] ✗ pred=0 gold=1 | 1.3s/ex, ETA: 0.2min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [div]  Andorra -> CLICK
  [2/10] ✗ pred=0 gold=1 | 1.3s/ex, ETA: 0.2min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/10] ✗ pre

## ABLATION 1 & 2: ZERO-SHOT  (HTML-only | Image-only)


In [ ]:
# Zero-shot with HTML only
zs_html_preds, zs_html_metrics = run_ablation(
    ds_test, "zero_shot_html_only", mode="zero_shot", n_examples=None, input_mode="html_only"
)

zs_html_pred_path = os.path.join(CONFIG["output_dir"], "zero_shot_html_only_predictions.json")
with open(zs_html_pred_path, "w") as f:
    json.dump(zs_html_preds, f, indent=2, default=str)

zs_html_metrics_path = os.path.join(CONFIG["output_dir"], "zero_shot_html_only_metrics.json")
with open(zs_html_metrics_path, "w") as f:
    json.dump(zs_html_metrics, f, indent=2)

print("\n=== Zero-Shot HTML-Only Metrics ===")
for k, v in zs_html_metrics.items():
    print(f"  {k}: {v}")



Running ablation: zero_shot_html_only (zero_shot, input=html_only)
Examples: 1019, Scoring: True

  [1/1019] ✓ pred=0 gold=0 | 0.8s/ex, ETA: 13.7min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [label]   -> CLICK
  [2/1019] ✗ pred=0 gold=1 | 0.8s/ex, ETA: 13.3min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/1019] ✗ pred=0 gold=2 | 0.8s/ex, ETA: 13.2min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [span]  TikTok Series -> CLICK
  [4/1019] ✗ pred=0 gold=3 | 0.8s/ex, ETA: 13.1min
  [5/1019] ✗ pred=0 gold=4 | 0.8s/ex, ETA: 13.1min
  [6/1019] ✗ pred=4 gold=5 | 0.8s/ex, ETA: 13.1min
  [7/1019] ✗ pred=0 gold=6 | 0.8s/ex, ETA: 13.0min
  [8/1019] ✗ pred=1 gold=0 | 0.8s/ex, ETA: 13.0min
  [9/1019] ✓ pred=1 gold=1 | 

In [ ]:
# Zero-shot with image only
zs_img_preds, zs_img_metrics = run_ablation(
    ds_test, "zero_shot_image_only", mode="zero_shot", n_examples=None, input_mode="image_only"
)

zs_img_pred_path = os.path.join(CONFIG["output_dir"], "zero_shot_image_only_predictions.json")
with open(zs_img_pred_path, "w") as f:
    json.dump(zs_img_preds, f, indent=2, default=str)

zs_img_metrics_path = os.path.join(CONFIG["output_dir"], "zero_shot_image_only_metrics.json")
with open(zs_img_metrics_path, "w") as f:
    json.dump(zs_img_metrics, f, indent=2)

print("\n=== Zero-Shot Image-Only Metrics ===")
for k, v in zs_img_metrics.items():
    print(f"  {k}: {v}")



Running ablation: zero_shot_image_only (zero_shot, input=image_only)
Examples: 1019, Scoring: True

  [1/1019] ✗ pred=6 gold=0 | 1.5s/ex, ETA: 25.3min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [button]  Search -> CLICK
    gold: [label]   -> CLICK
  [2/1019] ✗ pred=6 gold=1 | 1.5s/ex, ETA: 25.2min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [button]  Search -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/1019] ✗ pred=1 gold=2 | 1.5s/ex, ETA: 25.1min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [div]  Andorra -> CLICK
    gold: [span]  TikTok Series -> CLICK
  [4/1019] ✗ pred=1 gold=3 | 1.5s/ex, ETA: 25.1min
  [5/1019] ✗ pred=1 gold=4 | 1.5s/ex, ETA: 25.1min
  [6/1019] ✗ pred=1 gold=5 | 1.5s/ex, ETA: 25.1min
  [7/1019] ✓ pred=6 gold=6 | 1.5s/ex, ETA: 25.1min
  [8/1019] ✓ pred=0 gold=0 | 1.5s/ex, ETA: 25.1min
  [9/101

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Free VRAM: 24.3 GB


In [ ]:
## Few-shot image setup: downscale few-shot context images to 512px
## (test images remain full resolution; this only affects the 3 in-context examples)

FEWSHOT_IMG_SIZE = 512

fewshot_images_full = []
for op_type in ["CLICK", "TYPE", "SELECT"]:
    if op_type in fewshot_examples:
        img = fewshot_examples[op_type]["image"]
        img_s = downscale_image(img, FEWSHOT_IMG_SIZE)
        fewshot_images_full.append(img_s)
        print(f"  {op_type}: {img.size} → {img_s.size}")

print(f"\nReady: {len(fewshot_images_full)} few-shot context images prepared")

  CLICK: (1280, 5429) → (120, 512)
  TYPE: (1280, 5429) → (120, 512)
  CLICK: (1280, 5429) → (120, 512)
  TYPE: (1280, 5429) → (120, 512)
  SELECT: (1280, 2863) → (228, 512)

Ready: 3 few-shot context images prepared
  SELECT: (1280, 2863) → (228, 512)

Ready: 3 few-shot context images prepared


## ABLATION 3 & 4: FEW-SHOT  (HTML-only | Image-only)


In [ ]:
# Few-shot with HTML only
if "fewshot_images_full" not in globals():
    print("fewshot_images_full not found; preparing now...")
    FEWSHOT_IMG_SIZE = 512
    fewshot_images_full = []
    for op_type in ["CLICK", "TYPE", "SELECT"]:
        if op_type in fewshot_examples:
            img = fewshot_examples[op_type]["image"]
            fewshot_images_full.append(downscale_image(img, FEWSHOT_IMG_SIZE))
    print(f"Prepared {len(fewshot_images_full)} few-shot images")

fs_html_preds, fs_html_metrics = run_ablation(
    ds_test, "few_shot_html_only", mode="few_shot", n_examples=None,
    fewshot_prefix_str=fewshot_prefix,
    fewshot_prefix_nohtml_str=fewshot_prefix_nohtml,
    fewshot_images=fewshot_images_full,
    input_mode="html_only"
)

fs_html_pred_path = os.path.join(CONFIG["output_dir"], "few_shot_html_only_predictions.json")
with open(fs_html_pred_path, "w") as f:
    json.dump(fs_html_preds, f, indent=2, default=str)

fs_html_metrics_path = os.path.join(CONFIG["output_dir"], "few_shot_html_only_metrics.json")
with open(fs_html_metrics_path, "w") as f:
    json.dump(fs_html_metrics, f, indent=2)

print("\n=== Few-Shot HTML-Only Metrics ===")
for k, v in fs_html_metrics.items():
    print(f"  {k}: {v}")


Running ablation: few_shot_html_only (few_shot, input=html_only)
Examples: 1019, Scoring: True

  [1/1019] ✓ pred=0 gold=0 | 1.7s/ex, ETA: 28.2min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [label]   -> CLICK
  [2/1019] ✗ pred=0 gold=1 | 1.7s/ex, ETA: 28.1min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/1019] ✗ pred=0 gold=2 | 1.7s/ex, ETA: 28.2min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [span]  TikTok Series -> CLICK
  [4/1019] ✗ pred=0 gold=3 | 1.7s/ex, ETA: 28.2min
  [5/1019] ✗ pred=6 gold=4 | 1.7s/ex, ETA: 28.2min
  [6/1019] ✗ pred=0 gold=5 | 1.7s/ex, ETA: 28.3min
  [7/1019] ✗ pred=0 gold=6 | 1.7s/ex, ETA: 28.2min
  [8/1019] ✗ pred=1 gold=0 | 1.7s/ex, ETA: 28.2min
  [9/1019] ✓ pred=1 gold=1 | 1.

In [ ]:
# Few-shot with image only
if "fewshot_images_full" not in globals():
    print("fewshot_images_full not found; preparing now...")
    FEWSHOT_IMG_SIZE = 512
    fewshot_images_full = []
    for op_type in ["CLICK", "TYPE", "SELECT"]:
        if op_type in fewshot_examples:
            img = fewshot_examples[op_type]["image"]
            fewshot_images_full.append(downscale_image(img, FEWSHOT_IMG_SIZE))
    print(f"Prepared {len(fewshot_images_full)} few-shot images")

fs_img_preds, fs_img_metrics = run_ablation(
    ds_test, "few_shot_image_only", mode="few_shot", n_examples=None,
    fewshot_prefix_str=fewshot_prefix,
    fewshot_prefix_nohtml_str=fewshot_prefix_nohtml,
    fewshot_images=fewshot_images_full,
    input_mode="image_only"
)

fs_img_pred_path = os.path.join(CONFIG["output_dir"], "few_shot_image_only_predictions.json")
with open(fs_img_pred_path, "w") as f:
    json.dump(fs_img_preds, f, indent=2, default=str)

fs_img_metrics_path = os.path.join(CONFIG["output_dir"], "few_shot_image_only_metrics.json")
with open(fs_img_metrics_path, "w") as f:
    json.dump(fs_img_metrics, f, indent=2)

print("\n=== Few-Shot Image-Only Metrics ===")
for k, v in fs_img_metrics.items():
    print(f"  {k}: {v}")


Running ablation: few_shot_image_only (few_shot, input=image_only)
Examples: 1019, Scoring: True

  [1/1019] ✗ pred=4 gold=0 | 1.6s/ex, ETA: 27.3min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [span]  Romantic -> CLICK
    gold: [label]   -> CLICK
  [2/1019] ✗ pred=4 gold=1 | 1.6s/ex, ETA: 27.0min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [span]  Romantic -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/1019] ✗ pred=4 gold=2 | 1.6s/ex, ETA: 27.0min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [span]  Romantic -> CLICK
    gold: [span]  TikTok Series -> CLICK
  [4/1019] ✗ pred=5 gold=3 | 1.6s/ex, ETA: 26.8min
  [5/1019] ✓ pred=4 gold=4 | 1.6s/ex, ETA: 26.8min
  [6/1019] ✗ pred=4 gold=5 | 1.6s/ex, ETA: 26.7min
  [7/1019] ✗ pred=5 gold=6 | 1.6s/ex, ETA: 26.7min
  [8/1019] ✗ pred=4 gold=0 | 1.6s/ex, ETA: 26.7min
  [9/101

## ABLATION 5 & 6: HTML + Image  (Zero-Shot | Few-Shot)


In [ ]:
# Few-shot with HTML + Image
if "fewshot_images_full" not in globals():
    print("fewshot_images_full not found; preparing now...")
    FEWSHOT_IMG_SIZE = 512
    fewshot_images_full = []
    for op_type in ["CLICK", "TYPE", "SELECT"]:
        if op_type in fewshot_examples:
            img = fewshot_examples[op_type]["image"]
            fewshot_images_full.append(downscale_image(img, FEWSHOT_IMG_SIZE))
    print(f"Prepared {len(fewshot_images_full)} few-shot images")

fs_both_preds, fs_both_metrics = run_ablation(
    ds_test, "few_shot_html_image", mode="few_shot", n_examples=None,
    fewshot_prefix_str=fewshot_prefix,
    fewshot_prefix_nohtml_str=fewshot_prefix_nohtml,
    fewshot_images=fewshot_images_full,
    input_mode="both"
)

fs_both_pred_path = os.path.join(CONFIG["output_dir"], "few_shot_html_image_predictions.json")
with open(fs_both_pred_path, "w") as f:
    json.dump(fs_both_preds, f, indent=2, default=str)

fs_both_metrics_path = os.path.join(CONFIG["output_dir"], "few_shot_html_image_metrics.json")
with open(fs_both_metrics_path, "w") as f:
    json.dump(fs_both_metrics, f, indent=2)

print("\n=== Few-Shot HTML+Image Metrics ===")
for k, v in fs_both_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Zero-shot with HTML + Image
zs_both_preds, zs_both_metrics = run_ablation(
    ds_test, "zero_shot_html_image", mode="zero_shot", n_examples=None, input_mode="both"
)

zs_both_pred_path = os.path.join(CONFIG["output_dir"], "zero_shot_html_image_predictions.json")
with open(zs_both_pred_path, "w") as f:
    json.dump(zs_both_preds, f, indent=2, default=str)

zs_both_metrics_path = os.path.join(CONFIG["output_dir"], "zero_shot_html_image_metrics.json")
with open(zs_both_metrics_path, "w") as f:
    json.dump(zs_both_metrics, f, indent=2)

print("\n=== Zero-Shot HTML+Image Metrics ===")
for k, v in zs_both_metrics.items():
    print(f"  {k}: {v}")

## RESULTS: COMPARISON TABLE


In [ ]:
print("\n" + "="*100)
print("COMPARISON: HTML-ONLY vs IMAGE-ONLY vs HTML+IMAGE  ×  ZERO-SHOT vs FEW-SHOT")
print("="*100)

all_ablations = {
    "ZS  HTML-only":   zs_html_metrics,
    "ZS  Image-only":  zs_img_metrics,
    "ZS  HTML+Image":  zs_both_metrics,
    "FS  HTML-only":   fs_html_metrics,
    "FS  Image-only":  fs_img_metrics,
    "FS  HTML+Image":  fs_both_metrics,
}

key_metrics = ["element_accuracy", "operation_f1", "step_success_rate", "task_success_rate", "top3_accuracy"]

col_w = 16
print(f"\n{'Metric':<25}" + "".join(f"{name:>{col_w}}" for name in all_ablations))
print("-" * (25 + len(all_ablations) * col_w))
for metric in key_metrics:
    print(f"{metric:<25}" + "".join(f"{m.get(metric, 0):>{col_w}.4f}" for m in all_ablations.values()))

print(f"\n{'num_examples':<25}" + "".join(f"{m.get('num_examples', 0):>{col_w}}" for m in all_ablations.values()))
print(f"{'num_tasks':<25}" + "".join(f"{m.get('num_tasks', 0):>{col_w}}" for m in all_ablations.values()))
print(f"{'time (min)':<25}" + "".join(f"{m.get('total_time_seconds', 0)/60:>{col_w}.1f}" for m in all_ablations.values()))



COMPARISON: HTML-ONLY vs IMAGE-ONLY  ×  ZERO-SHOT vs FEW-SHOT

Metric                     ZS  HTML-only ZS  Image-only  FS  HTML-only FS  Image-only
-------------------------------------------------------------------------------------
element_accuracy                  0.1698         0.2267         0.1747         0.2218
operation_f1                      0.6770         0.7464         0.6519         0.7149
step_success_rate                 0.1698         0.2267         0.1747         0.2218
task_success_rate                 0.0070         0.0211         0.0141         0.0423
top3_accuracy                     0.4622         0.4926         0.4534         0.5064

num_examples                        1019           1019           1019           1019
num_tasks                            142            142            142            142
time (min)                          15.7           46.0           31.8           47.9


## Cell 17: DOWNLOAD HELPER


In [ ]:
import shutil
shutil.make_archive("/content/outputs_archive", "zip", CONFIG["output_dir"])
from google.colab import files
files.download("/content/outputs_archive.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>